# Lab 2 -  Feature Selection


In [1]:
import sys, random
import numpy as np
import warnings

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.12.3
NumPy   : 2.4.3
Seed    : 414


In [2]:
# ── Data Loading: F1 Results + Qualifying via Ergast API ─────────────────────
import requests
import pandas as pd
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
SEASONS   = [2022, 2023, 2024]
TARGET    = 'top10'
BASE_URL  = "https://api.jolpi.ca/ergast/f1"

# Engineered (new) features for Lab 2
ENGINEERED_FEATURES = [
    'driver_prev_grid',
    'driver_top10_rate_3',
    'driver_q3_avg_3',
    'grid_quali_interaction',
]

# ── Helpers ───────────────────────────────────────────────────────────────────
def _paginate(url_template: str, table_key: str, row_key: str) -> list:
    """Generic paginator for Ergast-like endpoints."""
    offset, limit, rows = 0, 100, []
    while True:
        url = f"{url_template}?limit={limit}&offset={offset}"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()['MRData']

        total = int(data['total'])
        rows.extend(data[table_key][row_key])
        offset += limit
        if offset >= total:
            break
    return rows


def time_to_sec(t: str) -> float | None:
    """Convert 'm:ss.mmm' string to total seconds."""
    if not t:
        return None
    try:
        m, s = t.split(':')
        return int(m) * 60 + float(s)
    except Exception:
        return None


# ── Loaders ───────────────────────────────────────────────────────────────────
def get_season_results(year: int) -> pd.DataFrame:
    rows = []
    for race in _paginate(f"{BASE_URL}/{year}/results.json", 'RaceTable', 'Races'):
        for r in race['Results']:
            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'circuit': race['Circuit']['circuitId'],
                'date': race['date'],
                'driver': r['Driver']['driverId'],
                'driver_name': f"{r['Driver']['givenName']} {r['Driver']['familyName']}",
                'constructor': r['Constructor']['constructorId'],
                'grid': int(r['grid']),
                'position': int(r['position']) if r['position'].isdigit() else np.nan,
                'points': float(r['points']),
                'status': r['status'],
                'laps': int(r['laps']),
            })
    return pd.DataFrame(rows)


def get_qualifying_results(year: int) -> pd.DataFrame:
    rows = []
    for race in _paginate(f"{BASE_URL}/{year}/qualifying.json", 'RaceTable', 'Races'):
        for r in race['QualifyingResults']:
            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'driver': r['Driver']['driverId'],
                'quali_pos': int(r['position']),
                'q1_time_sec': time_to_sec(r.get('Q1')),
                'q2_time_sec': time_to_sec(r.get('Q2')),
                'q3_time_sec': time_to_sec(r.get('Q3')),
            })
    return pd.DataFrame(rows)


# ── Load & merge ──────────────────────────────────────────────────────────────
print("Loading F1 data...")
results, qualis = [], []
for year in SEASONS:
    df_r = get_season_results(year)
    df_q = get_qualifying_results(year)
    print(f"  {year}: {len(df_r)} results | {len(df_q)} quali entries")
    results.append(df_r)
    qualis.append(df_q)

df_results = pd.concat(results, ignore_index=True)
df_quali = pd.concat(qualis, ignore_index=True)

df_all = df_results.merge(
    df_quali[['season', 'round', 'driver', 'quali_pos', 'q1_time_sec', 'q2_time_sec', 'q3_time_sec']],
    on=['season', 'round', 'driver'],
    how='left'
)

# ── Feature engineering with temporal guard ───────────────────────────────────
df_all['date'] = pd.to_datetime(df_all['date'])
df_all = df_all.sort_values(['driver', 'date', 'season', 'round']).reset_index(drop=True)

# Target is race outcome; allowed only as label, never as input feature
# (post-race columns position/points/status/laps are excluded from X)
df_all[TARGET] = (df_all['position'].fillna(99) <= 10).astype(int)

# 1) Lag feature
df_all['driver_prev_grid'] = df_all.groupby('driver')['grid'].shift(1)

# 2) Rolling top10 rate (STRICT temporal): shift(1) BEFORE rolling aggregation
df_all['driver_top10_rate_3'] = (
    df_all.groupby('driver')[TARGET]
          .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# 3) Rolling qualifying pace (STRICT temporal): shift(1) BEFORE rolling aggregation
df_all['driver_q3_avg_3'] = (
    df_all.groupby('driver')['q3_time_sec']
          .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# 4) Interaction feature
df_all['grid_quali_interaction'] = df_all['grid'] * df_all['quali_pos']

# Leakage checklist (per feature)
leakage_checklist = pd.DataFrame([
    {'feature': 'grid', 'uses_only_pre_race_info': True, 'note': 'Known before race start'},
    {'feature': 'quali_pos', 'uses_only_pre_race_info': True, 'note': 'From qualifying session'},
    {'feature': 'q1_time_sec', 'uses_only_pre_race_info': True, 'note': 'From qualifying session'},
    {'feature': 'q2_time_sec', 'uses_only_pre_race_info': True, 'note': 'From qualifying session'},
    {'feature': 'q3_time_sec', 'uses_only_pre_race_info': True, 'note': 'From qualifying session'},
    {'feature': 'driver_prev_grid', 'uses_only_pre_race_info': True, 'note': 'Lag feature shift(1)'},
    {'feature': 'driver_top10_rate_3', 'uses_only_pre_race_info': True, 'note': 'shift(1) then rolling mean'},
    {'feature': 'driver_q3_avg_3', 'uses_only_pre_race_info': True, 'note': 'shift(1) then rolling mean'},
    {'feature': 'grid_quali_interaction', 'uses_only_pre_race_info': True, 'note': 'Interaction of pre-race features'},
    {'feature': 'position/points/status/laps', 'uses_only_pre_race_info': False, 'note': 'POST-race: excluded from X'},
])

FEATURE_COLS = [
    'grid',
    'quali_pos',
    'q1_time_sec',
    'q2_time_sec',
    'q3_time_sec',
    'driver_prev_grid',
    'driver_top10_rate_3',
    'driver_q3_avg_3',
    'grid_quali_interaction',
]

MODEL_COLS = ['season', 'round', 'race_name', 'driver', 'driver_name', 'date', TARGET] + FEATURE_COLS
df_model = df_all[MODEL_COLS].copy()

print(f"\nDataset final: {len(df_model)} filas")
print("NaNs por feature:")
print(df_model[FEATURE_COLS].isna().sum().sort_values(ascending=False))
print("\nLeakage checklist:")
print(leakage_checklist)

Loading F1 data...
  2022: 440 results | 440 quali entries
  2023: 440 results | 440 quali entries
  2024: 479 results | 479 quali entries

Dataset final: 1359 filas
NaNs por feature:
q3_time_sec               698
driver_q3_avg_3           417
q2_time_sec               348
driver_top10_rate_3        28
driver_prev_grid           28
q1_time_sec                13
grid                        0
quali_pos                   0
grid_quali_interaction      0
dtype: int64

Leakage checklist:
                       feature  uses_only_pre_race_info  \
0                         grid                     True   
1                    quali_pos                     True   
2                  q1_time_sec                     True   
3                  q2_time_sec                     True   
4                  q3_time_sec                     True   
5             driver_prev_grid                     True   
6          driver_top10_rate_3                     True   
7              driver_q3_avg_3           

In [3]:
df_all

,season,round,race_name,circuit,date,driver,driver_name,constructor,grid,position,points,status,laps,quali_pos,q1_time_sec,q2_time_sec,q3_time_sec,top10,driver_enc
0,2022,1,Bahrain Grand Prix,bahrain,2022-03-20,leclerc,Charles Leclerc,ferrari,1,1,26.0,Finished,57,1,91.471,90.932,90.558,True,13
1,2022,1,Bahrain Grand Prix,bahrain,2022-03-20,sainz,Carlos Sainz,ferrari,3,2,18.0,Finished,57,3,91.567,90.787,90.687,True,22
2,2022,1,Bahrain Grand Prix,bahrain,2022-03-20,hamilton,Lewis Hamilton,mercedes,5,3,15.0,Finished,57,5,92.285,91.048,91.238,True,8
3,2022,1,Bahrain Grand Prix,bahrain,2022-03-20,russell,George Russell,mercedes,9,4,12.0,Finished,57,9,92.269,91.252,92.216,True,21
4,2022,1,Bahrain Grand Prix,bahrain,2022-03-20,kevin_magnussen,Kevin Magnussen,haas,7,5,10.0,Finished,57,7,91.955,91.461,91.808,True,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1354,2024,24,Abu Dhabi Grand Prix,yas_marina,2024-12-08,kevin_magnussen,Kevin Magnussen,haas,14,16,0.0,Lapped,57,15,83.632,83.877,NaN,False,10
1355,2024,24,Abu Dhabi Grand Prix,yas_marina,2024-12-08,lawson,Liam Lawson,rb,12,17,0.0,Retired,55,12,83.733,83.472,NaN,False,12
1356,2024,24,Abu Dhabi Grand Prix,yas_marina,2024-12-08,bottas,Valtteri Bottas,sauber,9,18,0.0,Retired,30,9,83.481,83.341,83.204,False,3
1357,2024,24,Abu Dhabi Grand Prix,yas_marina,2024-12-08,colapinto,Franco Colapinto,williams,20,19,0.0,Retired,26,19,83.912,NaN,NaN,False,4


## Feature selection

### Selected features (with temporal guard)

Base pre-race features:
- `grid`, `quali_pos`, `q1_time_sec`, `q2_time_sec`, `q3_time_sec`

Engineered features (new for this lab):
- `driver_prev_grid` (lag with `shift(1)`)
- `driver_top10_rate_3` (rolling mean of prior 3 races, computed as `shift(1)` then `rolling`)
- `driver_q3_avg_3` (rolling qualifying pace with `shift(1)` before aggregation)
- `grid_quali_interaction` (interaction feature)

Leakage guard:
- `position`, `points`, `laps`, `status` are post-race variables and are excluded from model inputs.
- Validation is season-based (train past seasons, test future season), without shuffling.

In [3]:
# ── Logistic Regression: Top 10 Predictor (Temporal Validation) ──────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

import warnings
warnings.filterwarnings('ignore')

TEST_SEASON = 2024


def add_driver_encoding(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit driver encoding on train only to avoid leakage."""
    train_df = train_df.copy()
    test_df = test_df.copy()

    driver_to_id = {driver: idx for idx, driver in enumerate(sorted(train_df['driver'].unique()))}
    train_df['driver_enc'] = train_df['driver'].map(driver_to_id).astype(int)
    test_df['driver_enc'] = test_df['driver'].map(driver_to_id).fillna(-1).astype(int)

    return train_df, test_df


def fill_missing_with_train_stats(train_df: pd.DataFrame, test_df: pd.DataFrame, feature_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Impute NaNs with train medians only."""
    train_df = train_df.copy()
    test_df = test_df.copy()

    medians = train_df[feature_cols].median(numeric_only=True)
    train_df[feature_cols] = train_df[feature_cols].fillna(medians)
    test_df[feature_cols] = test_df[feature_cols].fillna(medians)

    return train_df, test_df


def build_pipeline() -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            class_weight='balanced',
            max_iter=1000,
            random_state=RANDOM_SEED,
        ))
    ])


def temporal_split(df: pd.DataFrame, test_season: int = TEST_SEASON) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = df[df['season'] < test_season].copy()
    test_df = df[df['season'] == test_season].copy()

    if train_df.empty or test_df.empty:
        raise ValueError('Temporal split failed: train or test set is empty.')

    return train_df, test_df


def walk_forward_validation(df: pd.DataFrame, feature_cols: list[str], min_train_seasons: int = 1) -> pd.DataFrame:
    """Season-based walk-forward validation (no shuffling)."""
    seasons = sorted(df['season'].unique())
    rows = []

    for i in range(min_train_seasons, len(seasons)):
        train_seasons = seasons[:i]
        val_season = seasons[i]

        train_fold = df[df['season'].isin(train_seasons)].copy()
        val_fold = df[df['season'] == val_season].copy()

        train_fold, val_fold = add_driver_encoding(train_fold, val_fold)
        fold_features = feature_cols + ['driver_enc']
        train_fold, val_fold = fill_missing_with_train_stats(train_fold, val_fold, fold_features)

        X_train = train_fold[fold_features]
        y_train = train_fold[TARGET]
        X_val = val_fold[fold_features]
        y_val = val_fold[TARGET]

        pipeline = build_pipeline()
        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_val)
        y_proba = pipeline.predict_proba(X_val)[:, 1]

        rows.append({
            'train_seasons': str(train_seasons),
            'val_season': int(val_season),
            'accuracy': accuracy_score(y_val, y_pred),
            'precision': precision_score(y_val, y_pred, zero_division=0),
            'recall': recall_score(y_val, y_pred, zero_division=0),
            'f1': f1_score(y_val, y_pred, zero_division=0),
            'roc_auc': roc_auc_score(y_val, y_proba),
        })

    return pd.DataFrame(rows)


def build_error_analysis(test_df: pd.DataFrame, y_pred: np.ndarray, y_proba: np.ndarray) -> pd.DataFrame:
    """Return at least 3 concrete failure cases with hypothesis and next step."""
    errors = test_df.copy()
    errors['y_true'] = errors[TARGET].astype(int)
    errors['y_pred'] = y_pred.astype(int)
    errors['p_top10'] = y_proba

    errors = errors[errors['y_true'] != errors['y_pred']].copy()
    if errors.empty:
        return pd.DataFrame(columns=['season', 'round', 'race_name', 'driver_name', 'error_type', 'p_top10', 'hypothesis', 'next_step'])

    errors['error_type'] = np.where((errors['y_true'] == 1) & (errors['y_pred'] == 0), 'FN', 'FP')
    errors['model_confidence'] = np.where(errors['y_pred'] == 1, errors['p_top10'], 1 - errors['p_top10'])
    errors = errors.sort_values('model_confidence', ascending=False).head(3).copy()

    def infer_hypothesis(row: pd.Series) -> str:
        if row['error_type'] == 'FN' and row['grid'] > 10:
            return 'Recovery drives from deep grid are under-modeled.'
        if row['error_type'] == 'FP' and row['grid'] <= 5:
            return 'Front-grid starts are over-trusted without race-pace signal.'
        if row['driver_enc'] == -1:
            return 'Unseen driver encoding (-1) reduces generalization.'
        return 'Current features miss race-pace/context factors for this case.'

    def infer_next_step(row: pd.Series) -> str:
        if row['driver_enc'] == -1:
            return 'Add constructor-level history features for unseen drivers.'
        if row['error_type'] == 'FN':
            return 'Add overtaking trend and prior race-pace proxies.'
        return 'Add circuit-specific interaction and weather proxies.'

    errors['hypothesis'] = errors.apply(infer_hypothesis, axis=1)
    errors['next_step'] = errors.apply(infer_next_step, axis=1)

    return errors[
        ['season', 'round', 'race_name', 'driver_name', 'error_type', 'p_top10', 'hypothesis', 'next_step']
    ].reset_index(drop=True)


# ── Train & evaluate with temporal split ──────────────────────────────────────
def train_and_evaluate_temporal(df: pd.DataFrame) -> dict:
    train_df, test_df = temporal_split(df, test_season=TEST_SEASON)

    # Add encoding and imputation using train-only statistics
    train_df, test_df = add_driver_encoding(train_df, test_df)
    feature_cols = FEATURE_COLS + ['driver_enc']
    train_df, test_df = fill_missing_with_train_stats(train_df, test_df, feature_cols)

    X_train = train_df[feature_cols]
    y_train = train_df[TARGET].astype(int)
    X_test = test_df[feature_cols]
    y_test = test_df[TARGET].astype(int)

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }

    print(f"Train seasons: {sorted(train_df['season'].unique())}")
    print(f"Test season : {TEST_SEASON}")
    print("\n── Holdout metrics (temporal split) ───────────────────────")
    for k, v in metrics.items():
        print(f"{k:>9}: {v:.3f}")

    print("\n── Classification report (test season) ───────────────────")
    print(classification_report(y_test, y_pred, target_names=['No Top10', 'Top10']))

    # Optional walk-forward validation for temporal stability
    wf = walk_forward_validation(df, FEATURE_COLS)
    print("\n── Walk-forward by season (no shuffle) ───────────────────")
    print(wf)

    # Error analysis: at least 3 failure modes with race/driver examples
    error_cases = build_error_analysis(test_df, y_pred, y_proba)
    print("\n── Error analysis (top 3 confident mistakes) ─────────────")
    print(error_cases)

    return {
        'pipeline': pipeline,
        'metrics': metrics,
        'walk_forward': wf,
        'error_cases': error_cases,
        'feature_cols': feature_cols,
    }


# ── Run ───────────────────────────────────────────────────────────────────────
results = train_and_evaluate_temporal(df_model)

Train seasons: [np.int64(2022), np.int64(2023)]
Test season : 2024

── Holdout metrics (temporal split) ───────────────────────
 accuracy: 0.837
precision: 0.843
   recall: 0.829
       f1: 0.836
  roc_auc: 0.910

── Classification report (test season) ───────────────────
              precision    recall  f1-score   support

    No Top10       0.83      0.85      0.84       239
       Top10       0.84      0.83      0.84       240

    accuracy                           0.84       479
   macro avg       0.84      0.84      0.84       479
weighted avg       0.84      0.84      0.84       479


── Walk-forward by season (no shuffle) ───────────────────
                      train_seasons  val_season  accuracy  precision  \
0                  [np.int64(2022)]        2023  0.768182   0.770642   
1  [np.int64(2022), np.int64(2023)]        2024  0.837161   0.843220   

     recall        f1   roc_auc  
0  0.763636  0.767123  0.827748  
1  0.829167  0.836134  0.910077  

── Error analysis (t